**DATA QUALITY AND VALIDATION IN ETL**

**Question 1:** Define Data Quality in the context of ETL Pipelines. Why is it more than just data cleaning.

**Answer:** Data Quality in ETL Pipelines refers to the degree at which data is fit for its intended use in downstream processes ( reporting, analytics, ML models, decision-making). It is measured by several dimensions:

1. Accuracy — Data correctly represents real-world values.
2. Completeness — No missing or null values where required.
3. Consistency — Same data is represented uniformly across the system.
4. Timeliness — Data is up-to-date and available when needed.
5. Uniqueness — No unwanted duplicates.
6. Validity — Data conforms to defined rules, formats, ranges, and business logic.
7. Integrity — Relationships between datasets (e.g., referential integrity) are maintained.

Why is it more than just data cleaning:-

Data cleaning is only one part of data quality (mainly fixing missing values, correcting formats, removing obvious errors). Data Quality is a broader, ongoing process that includes prevention, validation, monitoring, and governance. It involves enforcing business rules, profiling data, validating during transformation, handling referential integrity, and continuously monitoring quality metrics — not just reactive cleaning after problems occur.

**Question 2:** Explain why poor data quality leads to misleading dashboards and incorrect decisions.

**Answer:** Poor Quality data directly impacts the reliabiltiy of insights derived from it:

1. *Misleading Dashboards:* Aggregations (SUM, AVG, COUNT) become incorrect due to duplicates, missing values, or wrong data. For example, revenue may be overstated because of duplicate transactions.

2. *Garbage In, Garbage Out (GIGO):* Analytics and BI tools cannot produce trustworthy outputs if input data is flawed.

3. *Incorrect Business Decisions:* Management may overstock products, misallocate marketing budgets, or wrongly assess customer behavior based on distorted numbers.

4. *Loss of Trust:* Repeated inaccurate reports reduce stakeholder confidence in the entire analytics system.

5. *Financial & Reputational Risk:* Wrong decisions can lead to monetary losses, compliance issues, or damaged customer trust.

**Question 3:** Explain Duplicate data ? Explain three causes in ETL Pipelines.

**Answer:** Duplicate refer to multiple records that represent the same real world entity or transaction in a dataset.

Three common causes in ETL pipelines:

1. *Source System Issues:* The source application itself generates duplicates (e.g., same transaction recorded multiple times due to retry logic, network failure, or double submission by users).

2. *Incremental Load without Proper Delta Logic:* ETL jobs pull the same data again in subsequent runs because of incorrect watermarking, missing CDC (Change Data Capture), or faulty incremental keys.

3. *Join or Union Operations without Deduplication:* When combining data from multiple sources or tables (e.g., union of daily files), records get duplicated if there is no proper deduplication step using business keys.

Other causes include lack of unique constraints, poor merging logic, or reprocessing failed batches without idempotency.


**Question 4:** Differentiate between exact, partial and fuzzy duplicates.

**Answer:**


*   *Exact Duplicates:* Records that are 100% identical across all fields (or across the defined business key). They are byte-for-byte the same.
- Example: Rows 201, 203, and 208 in the given dataset are exact duplicates.
*   *Partial Duplicates:* Records that match on some key fields (business key) but differ in other attributes (e.g., same customer and transaction but different city spelling or timestamp variation).



*   *Fuzzy Duplicates:* Records that refer to the same entity but do not match exactly due to variations in spelling, abbreviations, typos, or formatting.
    -   Detected using similarity algorithms (Levenshtein distance, Soundex, Jaro-Winkler, etc.).
    -   Example: "Rahul Mehta", "Rahul K. Mehta", "R. Mehta" — all refer to the same person.





**Question 5:** Why should Data validation be performed during transformation rather than after loading ?

**Answer:**

*   *Early Detection & Correction :* Catching issues during transformation allows fixing or rejecting bad records before they enter the target warehouse, preventing pollution of the final dataset.

*   *Better Performance:* Invalid data is filtered early, reducing load volume and avoiding costly post-load cleanup.



*   *Data Integrity :* Target tables can maintain stricter constraints (e.g., NOT NULL, foreign keys) if only clean data is loaded.

*   *Auditability & Traceability:* Validation during transformation allows logging of rejected records with reasons, making debugging easier.



*   *Business Rule Enforcement :* Complex validations (cross-field, referential, business logic) are easier and more efficient to apply in the transformation layer (Spark, dbt, Informatica, etc.) than after loading.


**Question 6:** Explain how business rules help in validating data accuracy ? Give an example.

**Answer:** Business rules are domain-specific logic defined by business stakeholders that data must satisfy to be considered accurate and valid.
They go beyond technical rules (data type, format) and validate semantic correctness.
Example (using the Sales_Transactions dataset):



*   Business Rule: "Quantity must always be a positive integer greater than 0, and Txn_Amount must equal Quantity × Unit_Price (if unit price is known). Also, Txn_Date cannot be in the future or null for completed transactions."

Validation using thsi rule :

*   Row 205: Quantity = Null → Violates rule.
*   Row 206: Txn_Amount = Null → Violates rule.


*   Row 207: Txn_Date = Null → Violates rule

Such rules help detect logically incorrect data that technical validation alone would miss.













**Question 7:** Write an SQL query on Sales_Transactions to list all duplicate keys and their counts using the business key (Customer_ID + Product_ID + Txn_Date + Txn_Amount)

**Answer:** SELECT
    Customer_ID,
    Product_ID,
    Txn_Date,
    Txn_Amount,
    COUNT(*) AS duplicate_count


FROM Sales_Transactions
GROUP BY
    Customer_ID,
    Product_ID,
    Txn_Date,
    Txn_Amount
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC;

Expected output based on the dataset:



*   C101, P11, 2025-12-01, 4000 → 3


**Question 8:** Enforcing Referential Integrity.

**Answer:** Identify violating Customer_IDs:


*   From Sales_Transactions: C105 and C106 do not exist in the Customers_Master table.

SQL Query to detect referential integrity violations:

SELECT DISTINCT

    st.Customer_ID

FROM Sales_Transactions st
LEFT JOIN Customers_Master cm

    ON st.Customer_ID = cm.CustomerID
WHERE cm.CustomerID IS NULL;

Alternative (using NOT EXISTS - often more performant):

SELECT DISTINCT Customer_ID

FROM Sales_Transactions st

WHERE NOT EXISTS (

    SELECT 1
    FROM Customers_Master cm
    WHERE cm.CustomerID = st.Customer_ID
);


This will return: C105, C106.



